In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset, load_from_disk
import re
from collections import Counter
import os
import json
from tqdm import tqdm
from pathlib import Path

# Mount Google Drive for Colab
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Configuration and Checkpoint Management for Google Colab
CHUNK_SIZE = 10000  # Process 5000 rows per chunk (adjust based on memory)

# Google Drive paths
DRIVE_BASE_PATH = "/content/drive/MyDrive"
OUTPUT_FILE = f"{DRIVE_BASE_PATH}/cleaned_lyrics.csv"  # Single output file (append mode)
CHECKPOINT_FILE = f"{DRIVE_BASE_PATH}/cleanup_checkpoint.json"  # Track progress

def load_checkpoint():
    """Load the last processed chunk index from checkpoint file"""
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, 'r') as f:
            checkpoint = json.load(f)
            return checkpoint.get('last_processed_chunk', -1), checkpoint.get('total_chunks', None)
    return -1, None

def save_checkpoint(chunk_index, total_chunks):
    """Save the current progress to checkpoint file"""
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump({
            'last_processed_chunk': chunk_index,
            'total_chunks': total_chunks,
            'output_file': OUTPUT_FILE
        }, f)

# Load checkpoint to know where to resume from
last_chunk, _ = load_checkpoint()
if last_chunk >= 0:
    print(f"✓ Checkpoint found! Will resume from chunk {last_chunk + 1}")
else:
    print(f"✓ Starting fresh processing")
    # Delete output file if starting fresh
    if os.path.exists(OUTPUT_FILE):
        os.remove(OUTPUT_FILE)

In [ ]:
# Load the dataset from Google Drive
print("Loading dataset from Google Drive...")

# Dataset file path on Google Drive
DATA_FILE = f"{DRIVE_BASE_PATH}/song_lyrics.csv"

# Load dataset
dataset = load_dataset("csv", data_files=DATA_FILE)
print(f"✓ Dataset loaded successfully!")
print(f"  Initial total rows: {len(dataset['train']):,}")
print(f"  (Final chunk calculation will happen after filtering)")

In [ ]:
# Move to the cleaning functions - these remain the same
def remove_section_tags(text):
    """Remove [section] tags like [Verse], [Chorus], etc."""
    return re.sub(r"\[.*?\]", "", text)

def remove_adlibs(text):
    """Remove parenthetical content like (ad-libs)"""
    return re.sub(r"\([^)]*\)", "", text)

def normalize_space(text):
    """Normalize whitespace while preserving line breaks"""
    lines = text.split('\n')
    normalized_lines = [" ".join(line.split()) for line in lines]
    return "\n".join(normalized_lines)

def remove_excess_repetition(lines, max_repeat=1):
    """
    Deduplicate repeated lines (useful for chorus control)
    """
    seen = Counter()
    cleaned = []
    for line in lines:
        if seen[line] < max_repeat:
            cleaned.append(line)
            seen[line] += 1
    return cleaned

def clean_lyrics_text(text):
    """
    Clean a single lyrics text - called for each row
    """
    try:
        # 1. Structural cleanup
        text = remove_section_tags(text)
        text = remove_adlibs(text)
        
        # 2. Whitespace normalization
        text = normalize_space(text)
        text = text.lower()
        
        # 3. Line split
        lines = [l.strip() for l in text.split("\n") if l.strip()]
        
        # 4. Remove excess repetition
        lines = remove_excess_repetition(lines, max_repeat=1)
        
        # 5. Rejoin into cleaned text
        cleaned_text = "\n".join(lines)
        return cleaned_text
    except Exception as e:
        print(f"Error cleaning lyrics: {e}")
        return text  # Return original if cleaning fails

In [ ]:
# Filter rows to keep only English songs (language_cld3 == "en")
print("Filtering dataset for English songs only...")
initial_count = len(dataset['train'])

# Filter the dataset to keep only rows where language_cld3 is "en"
dataset['train'] = dataset['train'].filter(lambda x: x['language_cld3'] == 'en')

filtered_count = len(dataset['train'])
removed_count = initial_count - filtered_count

print(f"✓ Language filtering complete:")
print(f"  Initial rows: {initial_count:,}")
print(f"  English rows: {filtered_count:,}")
print(f"  Removed rows: {removed_count:,} ({removed_count/initial_count*100:.2f}%)")

# Remove unnecessary columns after filtering
dataset['train'] = dataset['train'].remove_columns([
    "title", "tag", "artist", "year", "views", 
    "features", "id", "language_cld3", "language_ft", "language"
])

print("✓ Columns cleaned")

# IMPORTANT: Recalculate total_rows and total_chunks after filtering
total_rows = len(dataset['train'])
total_chunks = (total_rows + CHUNK_SIZE - 1) // CHUNK_SIZE

print(f"\n✓ Updated chunk calculation after filtering:")
print(f"  Total rows (filtered): {total_rows:,}")
print(f"  Total chunks: {total_chunks}")

# Update checkpoint with new total chunks
save_checkpoint(-1, total_chunks)

In [ ]:
# Main Processing Pipeline: Load, Clean, and Append to Single Output File
print("="*60)
print("CHUNK-BY-CHUNK PROCESSING WITH CHECKPOINTING")
print("="*60)

# Get checkpoint info
last_processed_chunk, total_chunks = load_checkpoint()
start_chunk = last_processed_chunk + 1

print(f"\nConfiguration:")
print(f"  • Chunk size: {CHUNK_SIZE} rows")
print(f"  • Output file: {OUTPUT_FILE}")
print(f"  • Starting from chunk: {start_chunk}")
print(f"  • Total chunks: {total_chunks}")
print(f"\nProcessing will start...\n")

try:
    # Process each chunk
    for chunk_idx in tqdm(range(start_chunk, total_chunks), desc="Overall Progress", unit="chunk"):
        try:
            # Calculate chunk boundaries
            start_idx = chunk_idx * CHUNK_SIZE
            end_idx = min((chunk_idx + 1) * CHUNK_SIZE, total_rows)
            
            # Load this specific chunk
            chunk_data = dataset['train'].select(range(start_idx, end_idx))
            
            # Convert to DataFrame for easier processing
            chunk_df = chunk_data.to_pandas()
            
            # Clean the lyrics in this chunk (without progress bar for individual chunk)
            chunk_df['clean_lyrics'] = chunk_df['lyrics'].apply(clean_lyrics_text)
            
            # Keep only the cleaned lyrics column
            chunk_df = chunk_df[['lyrics', 'clean_lyrics']]
            
            # Append this chunk to the single output CSV file
            if os.path.exists(OUTPUT_FILE):
                # Append without writing header
                chunk_df.to_csv(OUTPUT_FILE, mode='a', header=False, index=False)
            else:
                # First chunk - write with header
                chunk_df.to_csv(OUTPUT_FILE, mode='w', header=True, index=False)
            
            # Update checkpoint after successful save
            save_checkpoint(chunk_idx, total_chunks)
            
        except Exception as e:
            print(f"\n✗ Error processing chunk {chunk_idx}: {e}")
            print(f"  Checkpoint saved at chunk {chunk_idx - 1}. Resume processing to continue.")
            raise
    
    print("\n" + "="*60)
    print("✓ ALL CHUNKS PROCESSED SUCCESSFULLY!")
    print(f"✓ Total rows written to {OUTPUT_FILE}")
    print("="*60)
    
except KeyboardInterrupt:
    print("\n✗ Processing interrupted by user")
    print(f"  Last successfully processed chunk: {last_processed_chunk}")
    print(f"  Checkpoint saved. You can resume later.")
except Exception as e:
    print(f"\n✗ Processing failed: {e}")
    print(f"  Checkpoint saved. You can resume later.")

In [ ]:
# Check final output (memory-efficient version)
print("\n" + "="*60)
print("PROCESSING COMPLETE - FINAL STATUS")
print("="*60)

if os.path.exists(OUTPUT_FILE):
    # Get file size without loading into memory
    file_size_mb = os.path.getsize(OUTPUT_FILE) / (1024**2)
    
    # Count total rows efficiently without loading all data
    with open(OUTPUT_FILE, 'r') as f:
        total_rows = sum(1 for _ in f) - 1  # Subtract 1 for header
    
    # Read only first few rows to verify structure
    sample_df = pd.read_csv(OUTPUT_FILE, nrows=5)
    
    print(f"\n✓ Output file created: {OUTPUT_FILE}")
    print(f"  Total rows: {total_rows:,}")
    print(f"  Columns: {sample_df.columns.tolist()}")
    print(f"  File size: {file_size_mb:.2f} MB")
    print(f"\nFirst few rows:")
    print(sample_df)
else:
    print(f"✗ Output file not found: {OUTPUT_FILE}")

In [ ]:
# Utility: Check checkpoint status and cleanup
def show_processing_status():
    """Display the current processing status"""
    print("\n" + "="*60)
    print("PROCESSING STATUS")
    print("="*60)
    
    last_chunk, total_chunks = load_checkpoint()
    
    if total_chunks is None:
        print("No processing has started yet")
        return
    
    processed = last_chunk + 1
    remaining = total_chunks - processed
    percentage = (processed / total_chunks) * 100 if total_chunks > 0 else 0
    
    print(f"\nProgress:")
    print(f"  • Chunks processed: {processed}/{total_chunks} ({percentage:.1f}%)")
    print(f"  • Chunks remaining: {remaining}")
    print(f"  • Last processed chunk: {last_chunk}")
    
    # Check output file (memory-efficient)
    if os.path.exists(OUTPUT_FILE):
        # Get file size without loading into memory
        file_size_mb = os.path.getsize(OUTPUT_FILE) / (1024**2)
        
        # Count total rows efficiently without loading all data
        with open(OUTPUT_FILE, 'r') as f:
            total_rows = sum(1 for _ in f) - 1  # Subtract 1 for header
        
        print(f"\nOutput File:")
        print(f"  • File: {OUTPUT_FILE}")
        print(f"  • Total rows: {total_rows:,}")
        print(f"  • File size: {file_size_mb:.2f} MB")
    else:
        print(f"\nOutput File:")
        print(f"  • File not yet created")

def reset_checkpoint():
    """Reset checkpoint to start from the beginning"""
    if os.path.exists(CHECKPOINT_FILE):
        os.remove(CHECKPOINT_FILE)
        if os.path.exists(OUTPUT_FILE):
            os.remove(OUTPUT_FILE)
        print("✓ Checkpoint reset. Processing will start from the beginning.")
        print(f"✓ Deleted output file: {OUTPUT_FILE}")
    else:
        print("No checkpoint file found.")

# Display status
show_processing_status()